# 支线 12：Mid-Training、退火与持续预训练 — 预训练完了还不算完

> **前情回顾**：你已经知道怎么训练一个 LLM（Part 5）、缩放定律告诉你训多大（Part 9）、数据怎么准备（Part 10）、LoRA 省显存微调（Part 11）。
>
> **本 Part 目标**：理解 2024 年最重要的训练策略——**预训练不是终点，而是起点。**

核心问题：
1. 训练到一半想续训怎么办？→ **WSD 调度器**
2. 训练最后阶段 loss 怎么「冲刺」？→ **退火（Annealing）**
3. 怎么把通用模型变成领域专家？→ **持续预训练（CPT）**
4. 学新东西怎么不忘旧的？→ **LoRA 冻结 + 数据混合**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. 一个问题引出整个 Part

假设你用 Cosine 学习率调度器训了一个 7B 模型，训了 1T tokens。
训完后你发现：如果再加 500B tokens，loss 应该还能降。

**但问题来了：**

```
Cosine 调度器要求你提前说好总步数 T。
你已经训到 T 了，LR 已经降到接近 0。
现在想续训？LR 已经「没油了」——
就像车开到终点才发现油站还有油，但车已经熄火了。
```

这就是传统 Cosine 调度器的致命伤。
2024 年 MiniCPM 团队提出了一个解决方案：**WSD 调度器。**

## 2. WSD 调度器：让训练「随时可以续」

### 2.1 三阶段：热身 → 恒定 → 退火

WSD 把训练分成三段：

```
         Warmup    │     Stable (恒定)    │  Decay (退火)
  η_max ────┐     │                      │
            /      │                      │
           /       │                      │    \
          /        │                      │     \
  0 ─────┘         │                      │      ──
       前 5%       │      中间 80-85%     │   最后 10-15%
```

| 阶段 | LR 变化 | 用时 | 做什么 |
|:---|:---|:---|:---|
| Warmup | 0 → 最大值 | 前 5% 步数 | 让模型「热身」，别一上来就大步走 |
| Stable | **恒定不变** | 中间 80-85% | **满速训练**，随时可停 |
| Decay | 最大值 → 0 | 最后 10-15% | 「退火」，精细调整 |

核心创新在 **Stable 阶段**：LR 恒定不变 → 训练的任何时候都是「满血」状态。
这意味着：
- 你可以从 Stable 阶段的任何一个 checkpoint 开始做 Decay
- 你可以无限续训——想训多久训多久，最后做一个 Decay 收尾
- 你不需要提前知道总训练量

In [ ]:
# === Cosine vs WSD 手算对比 ===
print("=== Cosine vs WSD：关键位置的 LR 对比 ===")
print()

T = 1000
warmup = 50
stable_end = 850
eta_max = 0.01

def cosine_lr(t):
    """Cosine: warmup 后立刻开始降"""
    if t < warmup:
        return eta_max * t / warmup
    progress = (t - warmup) / (T - warmup)
    return eta_max * 0.5 * (1 + np.cos(np.pi * progress))

def wsd_lr(t):
    """WSD: warmup → stable(恒定) → decay"""
    if t < warmup:
        return eta_max * t / warmup
    elif t < stable_end:
        return eta_max  # 恒定！
    else:
        progress = (t - stable_end) / (T - stable_end)
        return eta_max * (0.001 ** progress)

print(f"{'步数':>6s}  {'阶段':>8s}  {'Cosine LR':>12s}  {'WSD LR':>12s}  {'差异'}")
print("-" * 60)

for t in [0, 50, 200, 500, 700, 850, 900, 950, 999]:
    if t < warmup:
        phase = "Warmup"
    elif t < stable_end:
        phase = "Stable"
    else:
        phase = "Decay"
    
    cos = cosine_lr(t)
    wsd = wsd_lr(t)
    diff = f"WSD 是 Cosine 的 {wsd/cos:.1f}x" if cos > 0 else "—"
    print(f"{t:>6d}  {phase:>8s}  {cos:>12.6f}  {wsd:>12.6f}  {diff}")

print()
print("关键观察：")
print("  第 500 步：Cosine 已经降到 0.004，WSD 还是满血 0.01")
print("  第 700 步：Cosine 只剩 0.001，WSD 依然满血")
print("  → Cosine 的训练越来越慢，WSD 一直在全速前进")
print()
print("结论：想续训？WSD 随时可以。Cosine？LR 已经降到地板了。")

In [ ]:
# === 可视化：Cosine vs WSD ===
steps = np.arange(T)
cos_lrs = [cosine_lr(t) for t in steps]
wsd_lrs = [wsd_lr(t) for t in steps]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：正常训练
ax = axes[0]
ax.plot(steps, cos_lrs, 'b-', lw=1.5, label='Cosine')
ax.plot(steps, wsd_lrs, 'r-', lw=1.5, label='WSD')
ax.axvspan(0, warmup, alpha=0.1, color='green', label='Warmup')
ax.axvspan(warmup, stable_end, alpha=0.05, color='blue', label='Stable')
ax.axvspan(stable_end, T, alpha=0.1, color='red', label='Decay')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine vs WSD')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 右图：续训场景
ax = axes[1]
ext = np.arange(600, 1200)
cos_ext = [cosine_lr(min(t, T-1)) for t in ext]  # Cosine 已经降到 0
wsd_ext = [wsd_lr(t) if t < T else eta_max * (0.001 ** ((t - stable_end) / 200)) for t in ext]
ax.plot(ext, cos_ext, 'b-', lw=1.5, label='Cosine (想续训)')
ax.plot(ext, wsd_ext, 'r-', lw=1.5, label='WSD (续训)')
ax.axvline(x=600, color='green', lw=1.5, label='续训起点')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('续训场景：Cosine「没油了」')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("左：WSD 在 85% 的时间里保持满速")
print("右：续训时 Cosine 几乎没有 LR，WSD 依然全速前进")

### 2.2 过度训练 + WSD = 2024 标准配方

回忆 Part 9 学的：LLaMA 3 8B 用了 D/N = 1875x 的数据（Chinchilla 说是 20x 就够了）。
这叫「过度训练」——小模型喂数据，推理时省大钱。

**但 Cosine 不支持无限过度训练。** 你得提前说好训多久。

WSD 解决了这个问题：

```
传统（Cosine）：
  确定数据量 → 确定训练步数 T → 按 T 设 Cosine → 训完拉倒

现代（WSD）：
  不设上限，WSD stable 一直训 → 想停就停 → 最后做 decay
  → 过度训练从「理论」变成了「可操作的工程实践」
```

| 模型 | 参数 N | 数据 D | D/N | 调度器 |
|:---|:---|:---|:---|:---|
| Chinchilla (2022) | 70B | 1.4T | 20x | Cosine |
| MiniCPM (2024) | 2.4B | ~1T | ~417x | **WSD** |
| LLaMA 3 8B (2024) | 8B | 15T | 1875x | Cosine (但业界在转向 WSD) |

---

## 3. 退火（Annealing）：为什么最后 10% 的训练最关键？

WSD 的 Decay 阶段又叫「退火」。这个名字来自冶金学：
- 金属加热后慢慢冷却，内部结构变得更有序、更坚固
- 训练也一样：LR 慢慢降低，模型参数「沉淀」到更优的位置

### 3.1 用打靶理解退火

```
Stable 阶段（LR 很大）:
  你在 100 米外打靶，子弹到处飞。
  有时偏左，有时偏右 → 命中点在靶心周围「震荡」。
  平均越来越近，但速度慢——因为你来回晃。

Decay 阶段（LR 在降）:
  你越走越近——50 米、30 米、10 米。
  距离近了，每颗子弹更精准。
  而且之前的大量练习已经让你知道靶心的大致方向，不再乱晃。
  → 命中率快速提高！
```

三句话总结：
1. **方向确定了** — Stable 阶段的大量更新让模型找到了正确方向，Decay 不再震荡
2. **步子变小了但更精准** — 每步改一点点，但每步都朝正确方向
3. **高质量数据在引导** — Decay 阶段混入好数据，像教练说「往左一点」

In [ ]:
# === 退火阶段 Loss 模拟 ===
print("=== 退火：Loss 在最后阶段猛降 ===")
print()

np.random.seed(42)
total_steps = 1000
stable_end = 850

steps = np.arange(total_steps)
loss = np.zeros(total_steps)

# Stable 阶段：loss 缓慢下降
for t in range(stable_end):
    progress = (t + 1) / stable_end
    loss[t] = 3.0 * (progress ** (-0.05)) + np.random.normal(0, 0.02)

# Decay 阶段：loss 加速下降（退火效果）
final_stable = loss[stable_end - 1]
for t in range(stable_end, total_steps):
    progress = (t - stable_end) / (total_steps - stable_end)
    drop = 0.15 * (1 - np.exp(-progress * 5))  # 额外下降
    loss[t] = final_stable - drop + np.random.normal(0, 0.01)

# 手算性价比
stable_avg = np.mean(loss[800:850])
decay_avg = np.mean(loss[-50:])
improvement = stable_avg - decay_avg

stable_per_step = (4.0 - stable_avg) / stable_end
decay_per_step = improvement / (total_steps - stable_end)

print(f"Stable 末期 loss: {stable_avg:.4f}")
print(f"Decay 末期 loss:  {decay_avg:.4f}")
print(f"退火带来改善:     {improvement:.4f} ({improvement/stable_avg*100:.1f}%)")
print()
print(f"Stable 每步改善: {stable_per_step:.6f}")
print(f"Decay  每步改善: {decay_per_step:.6f}")
print(f"退火每步效率是 Stable 的 {decay_per_step/stable_per_step:.1f}x")
print()
print("关键：退火只用了 15% 的步数，但每步的效率反而更高！")

In [ ]:
# === 退火可视化 ===
fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# 上图：完整 loss 曲线
ax = axes[0]
ax.plot(steps, loss, 'b-', lw=0.8, alpha=0.7)
ax.axvline(x=stable_end, color='red', ls='--', lw=1.5, label=f'Decay 开始 (step {stable_end})')
ax.axvspan(0, 50, alpha=0.1, color='green', label='Warmup')
ax.axvspan(50, stable_end, alpha=0.05, color='blue', label='Stable')
ax.axvspan(stable_end, 1000, alpha=0.1, color='red', label='Decay(退火)')
ax.set_ylabel('Loss')
ax.set_title('WSD 训练全过程 Loss')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 下图：放大 decay
ax = axes[1]
ax.plot(steps[750:], loss[750:], 'b-', lw=1.5)
ax.axvline(x=stable_end, color='red', ls='--', lw=2, label='Decay 开始')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('放大：退火阶段 Loss 明显加速下降')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("注意 decay 阶段（红色区域）的下降斜率明显变陡——这就是退火的威力。")

---

## 4. Mid-Training 数据策略：退火阶段该喂什么？

退火阶段不只是 LR 在降——**数据也在切换。**

直觉：就像考试前一周的「冲刺复习」——不再刷模拟卷（粗数据），而是看错题本和重点笔记（精数据）。

MiniCPM 的消融实验对比了三种策略：

| 策略 | Stable 阶段数据 | Decay 阶段数据 | 效果 |
|:---|:---|:---|:---|
| A: 不切换 | CC+Code+Books | 同左（粗数据） | baseline |
| B: 退火后 SFT | CC+Code+Books | 同 A，退火后再做 SFT | 较好 |
| C: **退火即混入** | CC+Code+Books | **CC+Code+Books + 高质量数据** | **最好** |

策略 C 为什么最好？
- 退火初期 LR 还比较大 → 模型有「余力」吸收新数据模式
- 退火后期 LR 很小 → 精细收敛，把新模式「焊死」在参数里
- 等于用 LR 递减的过程同步完成了「学习新内容 + 巩固旧知识」

In [ ]:
# === Decay 阶段数据配比手算 ===
print("=== 退火阶段数据配比方案 ===")
print()

# 场景：2.4B 模型，总共 1.2T tokens
# Stable: 1T（粗数据）  Decay: 200B（粗+精混合）
decay_total = 200  # B tokens

schemes = [
    ("不切换",        100,  0,  0,  0),
    ("保守混合",       70, 10, 10, 10),
    ("均衡混合",       50, 20, 15, 15),
    ("激进混合",       30, 30, 20, 20),
]

print(f"{'方案':<12s} {'粗数据':>8s} {'Wiki':>8s} {'SFT':>8s} {'指令':>8s} {'评价'}")
print("-" * 60)
for name, coarse, wiki, sft, inst in schemes:
    c = decay_total * coarse / 100
    w = decay_total * wiki / 100
    s = decay_total * sft / 100
    i = decay_total * inst / 100
    print(f"{name:<12s} {c:>6.0f}B  {w:>6.0f}B  {s:>6.0f}B  {i:>6.0f}B  ", end="")
    if coarse == 100:
        print("baseline")
    elif coarse >= 50:
        print("推荐")
    else:
        print("领域学到了，但遗忘风险高")

print()
print("黄金法则：")
print("  1. 粗数据（CC+Code）不能丢 → 防止遗忘预训练知识")
print("  2. 高质量数据占 30-50% → 让退火真正发挥「精调」效果")
print("  3. 高质量数据超 50% 会把模型「带偏」→ 忘记多样化知识")

In [ ]:
# === 数据配比可视化 ===
fig, ax = plt.subplots(figsize=(8, 4))

names = ['不切换', '保守混合', '均衡混合', '激进混合']
coarse = [100, 70, 50, 30]
quality = [0, 30, 50, 70]  # 高质量数据总和

x = np.arange(len(names))
ax.bar(x, coarse, label='粗数据 (CC+Code)', color='#95a5a6', alpha=0.8)
ax.bar(x, quality, bottom=coarse, label='高质量 (Wiki+SFT+指令)', color='#2ecc71', alpha=0.8)
ax.set_ylabel('占比 (%)')
ax.set_title('Decay 阶段数据配比')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.legend()
ax.axhline(y=50, color='red', ls='--', alpha=0.5, label='50% 红线')

# 标注推荐区间
ax.axvspan(0.5, 2.5, alpha=0.05, color='green')
ax.text(1.5, 55, '推荐区间', ha='center', fontsize=10, color='green')

plt.tight_layout()
plt.show()

---

## 5. 持续预训练（CPT）：把通用模型变成领域专家

退火和数据切换是「预训练内部」的策略。但还有一个更大的场景：

> **你已经有了一个通用 LLM（比如 LLaMA 3），现在想让它学会医疗/法律/金融的领域知识。**

这叫 **Continual Pretraining (CPT)**。

### 5.1 核心挑战：灾难性遗忘

```
你学了 3 年英语 → 考试 90 分（预训练）
然后只学 3 个月法语 → 法语 80 分，但英语掉到 50 分（灾难性遗忘）
如果同时复习英语 → 法语 80 分，英语 75 分（CPT 的目标）
```

用数据来理解：

In [ ]:
# === CPT 数据策略手算 ===
print("=== CPT 数据配比：领域数据 vs 通用数据 ===")
print()

cpt_total = 50  # B tokens

strategies = [
    ("纯领域数据",    100,   0, "遗忘严重：模型只会写病历"),
    ("领域为主",       70,  30, "通用能力部分退化"),
    ("均衡混合",       50,  50, "推荐：两不误"),
    ("保守混合",       30,  70, "领域学习不充分"),
]

print(f"总 CPT 数据: {cpt_total}B tokens")
print()
print(f"{'策略':<14s} {'领域':>8s} {'通用':>8s} {'评价'}")
print("-" * 50)
for name, domain, general, note in strategies:
    d = cpt_total * domain / 100
    g = cpt_total * general / 100
    print(f"{name:<14s} {d:>6.0f}B  {g:>6.0f}B  {note}")

print()
print("黄金法则：领域数据 : 通用数据 ≈ 1:1 到 2:1")
print("通用数据的作用 = 「锚」，防止模型忘记原来会的东西")

In [ ]:
# === 遗忘 vs 学习：甜蜜点可视化 ===
fig, ax = plt.subplots(figsize=(8, 4))

domain_pct = np.array([0, 30, 50, 70, 100])
forget_risk = np.array([0, 10, 25, 60, 90])  # 遗忘风险
domain_learn = np.array([0, 40, 75, 90, 95])  # 领域学习效果

ax2 = ax.twinx()
ax.plot(domain_pct, forget_risk, 'r-o', lw=2, ms=8, label='遗忘风险')
ax2.plot(domain_pct, domain_learn, 'b--s', lw=2, ms=8, label='领域学习')

ax.axvspan(40, 70, alpha=0.1, color='green', label='推荐区间')
ax.axvline(x=60, color='green', ls='--', alpha=0.7, label='最优 ~60%')

ax.set_xlabel('领域数据占比 (%)')
ax.set_ylabel('遗忘风险 (%)', color='red')
ax2.set_ylabel('领域学习效果 (%)', color='blue')
ax.set_title('CPT 甜蜜点：50-70% 领域数据')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='center right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("红色上升 = 遗忘越多，蓝色上升 = 学得越好")
print("交叉的绿色区域就是甜蜜点——两不误。")

### 5.2 LoRA 做 CPT：天然防遗忘 + 省显存

CPT 的两条路：

| 方法 | 可训练参数 | 防遗忘 | 显存 (7B) |
|:---|:---|:---|:---|
| 全量 CPT | 全部 7B | 差（所有参数都在变） | ~120 GB |
| **LoRA CPT** | ~21M (0.3%) | **好（原始权重冻结）** | **~15 GB** |

LoRA 天然防遗忘：原始权重 W 完全不动，只有小旁路 AB 在变。
等于在原来的地基上加一个小阁楼——地基完好无损。

In [ ]:
# === CPT 显存对比 ===
print("=== CPT 显存对比（7B 模型）===")
print()
print(f"{'方法':<20s} {'基座':>8s} {'训练参数':>10s} {'优化器':>10s} {'总计':>8s}")
print("-" * 60)

# 全量 CPT
print(f"{'全量 CPT':<20s} {'14 GB':>8s} {'7000M':>10s} {'84 GB':>10s} {'~120 GB':>8s}")

# LoRA CPT
print(f"{'LoRA CPT (r=16)':<20s} {'14 GB':>8s} {'~21M':>10s} {'~1 GB':>10s} {'~15 GB':>8s}")

# QLoRA CPT
print(f"{'QLoRA CPT (4bit)':<20s} {'3.5 GB':>8s} {'~21M':>10s} {'~1 GB':>10s} {'~6 GB':>8s}")

print()
print("QLoRA 只需要 6GB → 一张 RTX 3060 就能做 7B 模型的领域 CPT！")
print()
print("LoRA 做 CPT 的核心优势：")
print("  1. 原始权重冻结 → 地基不动 → 通用知识不丢失")
print("  2. 只训旁路 AB → 显存极省")
print("  3. 训完 merge → 推理零开销（和普通模型一样快）")
print()
print("注意：如果 CPT 数据量很大（>100B），LoRA 的低秩可能不够。")
print("这时可以考虑用更大的 r（如 64 或 128），或者直接全量 CPT。")

---

## 6. 全景：从预训练到 CPT 的完整训练策略

把本 Part 学的所有内容串成一条线：

```
┌─────────────────────────────────────────────────────────────────┐
│                   LLM 的完整训练生命周期                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. 预训练 (Pretraining)                                        │
│     数据：Common Crawl + Wiki + Code + Books (10T+ tokens)       │
│     调度器：WSD (stable 阶段，LR 恒定)                           │
│     ↓                                                           │
│                                                                 │
│  2. Mid-Training 退火 (Annealing)                               │
│     数据：混入 30-50% 高质量数据 (Wiki/SFT/指令)                  │
│     调度器：WSD decay 阶段，LR 递减                              │
│     ↓                                                           │
│                                                                 │
│  3. 持续预训练 (CPT) — 可选                                      │
│     场景：把通用模型适配到特定领域                                │
│     数据：领域 50-70% + 通用 30-50%                              │
│     方法：LoRA/QLoRA（省显存 + 防遗忘）                          │
│     ↓                                                           │
│                                                                 │
│  4. SFT + RLHF (后续 Part)                                      │
│     教模型「怎么回答问题」                                        │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

**每一步都是在上一步的基础上「加餐」，不是推翻重来。**

---

## 小结

确认你已经搞懂了这些：

- [ ] 1. Cosine 调度器的局限：训练步数提前定死，无法续训
- [ ] 2. WSD 三阶段：Warmup(热身) → Stable(恒定) → Decay(退火)
- [ ] 3. WSD 的核心优势：Stable 阶段 LR 恒定 → 随时可停，随时续训
- [ ] 4. 退火为什么有效：方向已确定 + 步子变小更精准 + 高质量数据引导
- [ ] 5. Mid-Training 数据策略：退火阶段混入 30-50% 高质量数据，效果最好
- [ ] 6. CPT 的核心挑战：灾难性遗忘——学新的忘旧的
- [ ] 7. CPT 数据黄金法则：领域 50-70% + 通用 30-50%，通用数据是「锚」
- [ ] 8. LoRA 做 CPT：冻结原始权重 = 地基不动 + 显存省到 1/8
- [ ] 9. 完整训练生命周期：预训练 → 退火 → CPT(可选) → SFT → RLHF

**一句话总结**：预训练不是终点，而是起点。WSD 让训练可以随时续，退火让最后 10% 的计算发挥最大效果，CPT 让通用模型变成领域专家——三者合在一起，就是 2024 年的训练标准配方。